# 10. Angka pada dashboard dan laporan

Menghitung ulang angka yang tampil di kartu ringkasan dan strip Temuan Kunci dashboard serta yang dikutip di laporan, supaya setiap angka bisa ditelusuri ke data.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
%matplotlib inline

## Kartu ringkasan halaman 1

In [2]:
import pandas as pd

MART = "../data/mart/"
lokasi = pd.read_csv(MART + "dim_lokasi.csv")
harian = pd.read_csv(MART + "fakta_udara_harian.csv", parse_dates=["tanggal"])
valid = harian[harian["hari_valid"]].merge(lokasi[["stasiun", "kategori_lokasi", "wilayah_gambut"]], on="stasiun")
episode = pd.read_csv(MART + "fakta_episode_asap.csv")
jaringan = pd.read_csv(MART + "fakta_jaringan_stasiun.csv")

kartu = {
    "Rata-rata PM2.5 harian (µg/m³)": round(valid["pm25"].mean(), 1),
    "Hari Tidak Sehat ke atas (%)": round(valid["kategori_ispu"].isin(["Tidak Sehat", "Sangat Tidak Sehat", "Berbahaya"]).mean() * 100, 1),
    "Episode kabut asap": len(episode),
    "Stasiun tidak andal": int((jaringan["andal"] != "Andal (≥ 90% data)").sum()),
}
pd.Series(kartu, name="Halaman 1 Ringkasan", dtype=object)

Rata-rata PM2.5 harian (µg/m³)    40.1
Hari Tidak Sehat ke atas (%)      16.7
Episode kabut asap                  93
Stasiun tidak andal                  6
Name: Halaman 1 Ringkasan, dtype: object

## Baku mutu tahunan, Marunda, dan Kototabang

In [3]:
tahunan = valid.groupby(["stasiun", valid["tanggal"].dt.year])["pm25"].mean().unstack()
print("Stasiun di atas 15 µg/m³ pada setiap tahun:", int(tahunan.gt(15).all(axis=1).sum()), "dari", len(tahunan))
print("Hari Marunda (JKT-02) di atas 55 µg/m³:", round((valid.loc[valid["stasiun"] == "JKT-02", "pm25"] > 55).mean() * 100, 1), "%")
asap = valid[valid["tanggal"].dt.month.isin([7, 8, 9, 10])]
print("Rata-rata Juli–Oktober Kototabang:", asap[asap["stasiun"] == "KTB-01"].groupby(asap["tanggal"].dt.year)["pm25"].mean().round(0).to_dict())
tahunan.round(1)

Stasiun di atas 15 µg/m³ pada setiap tahun: 22 dari 25
Hari Marunda (JKT-02) di atas 55 µg/m³: 100.0 %
Rata-rata Juli–Oktober Kototabang: {2024: 24.0, 2025: 34.0, 2026: 45.0}


tanggal,2024,2025,2026
stasiun,,,
BGL-01,25.6,25.7,25.4
BGR-01,13.2,13.2,13.3
BGR-02,19.2,19.2,19.3
BJB-01,43.0,60.0,48.1
BTM-01,43.3,48.0,52.5
IDP-01,9.2,9.2,9.7
JKT-01,48.0,48.1,48.2
JKT-02,60.0,60.0,60.2
JKT-03,54.0,54.0,54.1


## Jam puncak kawasan industri dan PM2.5 menurut kecepatan angin

In [4]:
jam = pd.read_csv(MART + "fakta_udara_per_jam.csv", usecols=["stasiun", "pm25", "jam", "wind_speed_ms"])
jam = jam.merge(lokasi[["stasiun", "kategori_lokasi"]], on="stasiun")
industri = jam[jam["kategori_lokasi"] == "Industri"].groupby("jam")["pm25"].mean()
print("Industri pukul 07.00:", round(industri[7], 1), "| pukul 13.00:", round(industri[13], 1),
      f"| selisih {industri[7] / industri[13] * 100 - 100:.0f}%")
jam["kelas_angin"] = pd.cut(jam["wind_speed_ms"], [0, 1, 2, 3, 4, 5, 20], right=False,
                            labels=["0–1", "1–2", "2–3", "3–4", "4–5", "≥5"])
jam.groupby(["kategori_lokasi", "kelas_angin"], observed=True)["pm25"].mean().unstack().round(1)

Industri pukul 07.00: 71.5 | pukul 13.00: 52.4 | selisih 37%


kelas_angin,0–1,1–2,2–3,3–4,4–5,≥5
kategori_lokasi,,,,,,
Gambut,48.0,53.4,49.1,45.7,44.4,NaN
Industri,54.7,60.3,57.7,49.4,51.4,NaN
Latar Bersih,13.4,13.8,14.4,15.2,11.9,16.6
Perkotaan,41.5,46.1,43.0,36.4,41.5,43.1
Tepi Jalan,27.9,30.8,38.5,48.4,54.4,NaN


## Kekuatan korelasi PM2.5 dengan cuaca

In [5]:
kor = pd.read_csv(MART + "fakta_korelasi_cuaca.csv")
nilai = kor[["rho_temperature_c", "rho_humidity_pct", "rho_wind_speed_ms"]].stack()
print("Jumlah nilai korelasi:", len(nilai), "| lemah (|ρ| < 0,2):", int((nilai.abs() < 0.2).sum()),
      "| sedang (0,2–0,4):", int(((nilai.abs() >= 0.2) & (nilai.abs() < 0.4)).sum()),
      "| ≥ 0,4:", int((nilai.abs() >= 0.4).sum()))

Jumlah nilai korelasi: 75 | lemah (|ρ| < 0,2): 66 | sedang (0,2–0,4): 6 | ≥ 0,4: 3


## Kasus ISPA balita saat episode asap di Palangka Raya dan Sampit

In [6]:
dampak = pd.read_csv(MART + "fakta_episode_dampak.csv")
pr = dampak[(dampak["stasiun"].isin(["KTG-01", "KTG-02"])) & (dampak["outcome"] == "resp_admissions_under5")].copy()
pr["rasio_episode_vs_biasa"] = (pr["rata2_hari_episode"] / pr["rata2_hari_biasa"]).round(2)
pr[["stasiun", "tahun", "hari_episode", "rata2_hari_episode", "rata2_hari_biasa", "rasio_episode_vs_biasa"]].round(2)

,stasiun,tahun,hari_episode,rata2_hari_episode,rata2_hari_biasa,rasio_episode_vs_biasa
0,KTG-01,2024,61,10.25,7.77,1.32
7,KTG-01,2025,70,14.99,7.77,1.93
14,KTG-01,2026,106,20.74,7.77,2.67
21,KTG-02,2024,61,10.07,7.61,1.32
28,KTG-02,2025,82,13.33,7.61,1.75
35,KTG-02,2026,106,20.58,7.61,2.70


## Baris data kesehatan yang dibuang

In [7]:
mentah = pd.read_csv("../data/raw/HLT-01_health_outcomes_daily.csv")
bersih = pd.read_csv(MART + "fakta_kesehatan_harian.csv")
kunci = set(zip(bersih["hospital_id"], bersih["tanggal"].astype(str).str[:10]))
dibuang = mentah[[(a, b[:10]) not in kunci for a, b in zip(mentah["hospital_id"], mentah["date"].astype(str))]]
print("Baris kesehatan mentah:", len(mentah), "| setelah dibersihkan:", len(bersih), "| dibuang:", len(dibuang))
dibuang[["date", "hospital_id"]]

Baris kesehatan mentah: 27491 | setelah dibersihkan: 27486 | dibuang: 5


,date,hospital_id
0,2023-12-31,RS-XXX-12
1,2023-12-31,RS-XXX-13
2,2023-12-31,RS-XXX-16
3,2023-12-31,RS-XXX-19
4,2023-12-31,RS-XXX-20
